# Create a Q&A Chatbot with LangChain Project

### Set the OpenAI API Key as an Environment Variable

In [1]:
%load_ext dotenv
%dotenv

### Import the Libraries

In [225]:
import pprint

# from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_pdf_inspector import PdfInspectorLoader

from langchain_text_splitters import (MarkdownHeaderTextSplitter, 
                                      TokenTextSplitter)

from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.messages import SystemMessage
from langchain_core.prompts import (PromptTemplate,
                                    HumanMessagePromptTemplate, 
                                    ChatPromptTemplate)
from langchain_core.runnables import (RunnablePassthrough, 
                                      RunnableLambda, 
                                      chain)

from langchain_openai import (ChatOpenAI, 
                              OpenAIEmbeddings)

# from langchain_chroma.vectorstores import Chroma
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

### Load the Course Transcript

In [3]:
loader_pdf = PdfInspectorLoader("Introduction_to_Tableau.pdf")

In [4]:
doc_list = loader_pdf.load()

In [5]:
string_list_concat = ''.join([doc.page_content for doc in doc_list])

In [6]:
string_list_concat = string_list_concat.replace('#  \n', '# ')

### Split the Course Transcript with MarkdownHeaderTextSplitter

In [123]:
headers_to_split_on = [
    ("#", "Section Title"),
    ("##", "Lecture Title"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers = False)

In [124]:
docs_list_md_split = md_splitter.split_text(string_list_concat)

### Create a Chain to Correct the Course Transcript

In [125]:
string_list_split = [doc.page_content for doc in docs_list_md_split]

In [126]:
PROMPT_FORMATTING_S = '''Improve the following Tableau lecture transcript by:
- Splitting the text into meaningful paragraphs
- Correcting any misplaced punctuation
- Fixing mistranscribed words (e.g., changing 'tableaux' to 'Tableau')"
'''

PROMPT_TEMPLATE_FORMATTING_H = '''This is the transcript:
{lecture_transcript}
'''

In [127]:
prompt_formatting_s = SystemMessage(content=PROMPT_FORMATTING_S)
prompt_template_formatting_h = HumanMessagePromptTemplate.from_template(PROMPT_TEMPLATE_FORMATTING_H)
chat_prompt_template_formatting = ChatPromptTemplate.from_messages([prompt_formatting_s,
                                                                   prompt_template_formatting_h])

In [128]:
chat = ChatOpenAI(model='gpt-6-luna', seed=365, temperature=1, max_completion_tokens=10000)

In [129]:
str_output_parser = StrOutputParser()

In [302]:
chain_formatting = chat_prompt_template_formatting | chat | str_output_parser

In [303]:
string_list_formatted = chain_formatting.batch([{'lecture_transcript': text} for text in string_list_split])

In [132]:
# Override the docs_list_md_split list such that the page_content parameter of each Document objects stores the updated lecture.

In [133]:
for idx, doc in enumerate(docs_list_md_split):
    docs_list_md_split[idx].page_content = string_list_formatted[idx]

In [134]:
docs_list_md_split

[Document(metadata={'Section Title': 'Introduction to Tableau', 'Lecture Title': 'Welcome to Tableau'}, page_content='# Introduction to Tableau\n\n## Welcome to Tableau\n\nHi, everyone. I’m Ned, and I’ll be your instructor for this course.\n\nTableau is an invaluable tool to learn on your journey to becoming a successful business intelligence analyst or data scientist. The art of these professions is storytelling: using data to tell stories and convince top management of the right course of action.\n\nBy completing this part of the program, you’ll know how to create charts and dashboards in Tableau. This is an essential step on your way to a data scientist role.'),
 Document(metadata={'Section Title': 'Introduction to Tableau', 'Lecture Title': 'Why use Tableau'}, page_content='## Why Use Tableau?\n\n### Make Your Data Make an Impact\n\nTableau has grown to become one of the most popular business intelligence tools in the entire world. It is BI software that allows non-technical users 

### Split the Lectures with TokenTextSplitter

In [135]:
token_splitter = TokenTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=500, chunk_overlap=50
)

Here we use `.split_documents` instead of `.split_text` in order to get documents instead of strings and to keep the metadata

In [136]:
docs_list_tokens_split = token_splitter.split_documents(docs_list_md_split)

In [137]:
len(docs_list_md_split)

25

In [138]:
len(docs_list_tokens_split)

42

In [139]:
len(string_list_split)

25

In [140]:
len(string_list_formatted)

25

In [141]:
len([chunks for chunks in docs_list_tokens_split])

42

### Create Embeddings, Vector Store, and Retriever

In [142]:
embedding = OpenAIEmbeddings(model='text-embedding-3-small')

In [150]:
# client = QdrantClient(path="./intro-to-tableau/")


# client.delete_collection(
#     collection_name="intro_to_tableau_collection"
# )






In [149]:
client.create_collection(
    collection_name="intro_to_tableau_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="intro_to_tableau_collection",
    embedding=embedding,
)

In [151]:
uuids = [str(uuid4()) for _ in range(len(docs_list_tokens_split))]

In [153]:
doc_ids = vector_store.add_documents(documents=docs_list_tokens_split, ids=uuids)

Verify what we get in the vectore_store. We should have the document with its metadata, text and vector.

In [154]:
record = client.retrieve(ids=[doc_ids[0]], 
                         collection_name='intro_to_tableau_collection', 
                         with_vectors=True)

In [ ]:
print(record)

In [156]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 2, "lambda_mult": 0.7})

In [157]:
runnable_retriever = RunnableLambda(lambda x: retriever.invoke(x))

### Create Prompts and Prompt Templates for the Q&A Chatbot Chain

In [166]:
PROMPT_CREATING_QUESTION = '''Lecture: {question_lecture}
Title: {question_title}
Body: {question_body}'''

PROMPT_RETRIEVING_S = '''You will receive a question from a student taking a Tableau course, which includes a title and a body. 
The corresponding lecture will also be provided.

Answer the question using only the provided context.

At the end of your response, include the section and lecture names where the context was drawn from, formatted as follows: 
Resources: 
Section: *Section Title*, Lecture: *Lecture Title* 
...
Replace *Section Title* and *Lecture Title* with the appropriate titles.'''

PROMPT_TEMPLATE_RETRIEVING_H = '''This is the question:
{question}

This is the context:
{context}'''

prompt_creating_question = PromptTemplate.from_template(PROMPT_CREATING_QUESTION)
prompt_retrieving_s = SystemMessage(content=PROMPT_RETRIEVING_S)
prompt_template_retrieving_h = HumanMessagePromptTemplate.from_template(PROMPT_TEMPLATE_RETRIEVING_H)

chat_prompt_template_retrieving = ChatPromptTemplate.from_messages([prompt_retrieving_s,
                                                                   prompt_template_retrieving_h
                                                                ])

### Create the First Version of the Q&A Chatbot Chain

In [306]:
chain_retrieving = ( prompt_creating_question 
                     | RunnableLambda(lambda x: x.text) 
                     | { 'context': runnable_retriever, 'question': RunnablePassthrough() }
                     | chat_prompt_template_retrieving 
                     | chat | str_output_parser
                   )

In [284]:
result = chain_retrieving.invoke({"question_lecture": "Adding a custom calculation",
                                  "question_title": "Why are we using SUM here? It's unclear to me.",
                                  "question_body": "This question refers to calculating the GM%."})

In [ ]:
result

### Create a Runnable Function to Format the Context

In [290]:
@chain
def format_context(dictionary):
    docs_as_string = ''
    docs = dictionary['context']
    for idx, doc in enumerate(docs):
        docs_as_string = docs_as_string + f'''
Document {idx + 1}
Section Title: {doc.metadata['Section Title']}
Lecture Title: {doc.metadata['Lecture Title']}
Content: {doc.page_content}
\n
-------------------
\n
'''
    return {**dictionary, 'context': docs_as_string}


In [292]:
chain_retrieving_improved = ( prompt_creating_question 
                     | RunnableLambda(lambda x: x.text) 
                     | { 'context': runnable_retriever, 'question': RunnablePassthrough() }
                     | format_context
                     | chat_prompt_template_retrieving 
                     | chat | str_output_parser
                   )

In [293]:
result_improved = chain_retrieving_improved.invoke({"question_lecture": "Adding a custom calculation",
                                                    "question_title": "Why are we using SUM here? It's unclear to me.",
                                                    "question_body": "This question refers to calculating the GM%."})

In [294]:
result_improved

'`SUM` is used because the table displays monthly totals. The GM% should be calculated as the month’s total gross profit divided by the month’s total revenue:\n\n`SUM([Gross Profit]) / SUM([Revenue])`\n\nFor January, that’s about 71 million ÷ 244 million, or 29%. Without `SUM`, Tableau isn’t calculating the ratio from each month’s total figures, which is why the result looks unexpectedly large. You can then format the calculated field as a percentage.\n\nResources:  \nSection: *Tableau Functionalities*, Lecture: *Adding a custom calculation*'

### Stream the Response

In [296]:
result_streamed = chain_retrieving_improved.stream({"question_lecture": "Adding a custom calculation",
                                                    "question_title": "Why are we using SUM here? It's unclear to me.",
                                                    "question_body": "This question refers to calculating the GM%."})

In [307]:
type(result_streamed)

generator

In [297]:
# Create a for-loop to stream the response
for i in result_streamed:
    print(i, end='')

`SUM` is used because the GM% should be calculated from the **monthly totals**: total gross profit divided by total revenue. For January, that means `71 million ÷ 244 million`, or about **29%**.

Use a calculated field like:

`SUM([Gross Profit]) / SUM([Revenue])`

Without `SUM`, Tableau isn’t dividing the total figures for each month, which is why the result looked unexpectedly large.

Resources:  
Section: *Tableau Functionalities*, Lecture: *Adding a custom calculation*